In [1]:
%load_ext autoreload
%autoreload 2

# Lesson Learn

## StateMachine
This idea using statemachine idea to iterative run until we met the end state

## State: with Enum
This idea serves as a blueprint what we can move to which state

## Data: with pydantic.BaseModel
This idea serves as a data spec that will be used in each state

## Method: with ABC @abstractmethod
This idea serves as a contract of each class

In [14]:
from enum import Enum
from pydantic import BaseModel, Field
from typing import Optional, List, Any
from abc import ABC, abstractmethod

class State(Enum):
    ROUTER = "router"
    CHAT = "chat"
    COMPLETE = "complete"

class DataContext(BaseModel):
    user_input: str
    answer: Optional[str] = Field(default=None)
    execution_trace: List[str] = Field(default_factory=list)  # Add this

class BaseState(ABC):
    @abstractmethod
    def execute(self, context: DataContext) -> State: pass

class StateRegistry:
    _states = {}
    
    @classmethod
    def register(cls, state_enum_value: State, state_class):  # Fix: remove BaseState type hint
        cls._states[state_enum_value] = state_class
    
    @classmethod
    def get(cls, state_enum_value: State):
        return cls._states[state_enum_value]()

def state(state_enum_value: State):
    def decorator(cls):
        StateRegistry.register(state_enum_value, cls)
        return cls
    return decorator

class StateMachine:
    def __init__(self, start_state: State, complete_state: State, context_class):  # Fix: remove type hint
        self.start_state = start_state
        self.complete_state = complete_state
        self.context_class = context_class
    
    def run(self, **initial_data) -> DataContext:  # Fix: return type
        context = self.context_class(**initial_data)
        current_state = self.start_state
        
        while current_state != self.complete_state:
            context.execution_trace.append(current_state.value)
            state_instance = StateRegistry.get(current_state)
            current_state = state_instance.execute(context)
        
        return context

# Usage:
@state(State.ROUTER)
class Router(BaseState):
    def execute(self, context: DataContext) -> State:
        if "help" in context.user_input.lower():
            return State.CHAT
        return State.CHAT

@state(State.CHAT)
class Chat(BaseState):
    def execute(self, context: DataContext) -> State:
        context.answer = f"Chat response: {context.user_input}"
        return State.COMPLETE

# Run it:
machine = StateMachine(State.ROUTER, State.COMPLETE, DataContext)
result = machine.run(user_input="hello world")
print(result.answer)  # "Chat response: hello world"
print(result.execution_trace)  # ['router', 'chat']


Chat response: hello world
['router', 'chat']


In [17]:
from enum import Enum
from pydantic import BaseModel, Field
from typing import Optional, List
from abc import ABC, abstractmethod

# Blueprint: States for both workflows
class State(Enum):
    ROUTER = "router"
    CHAT = "chat"
    GENERATE_SQL = "generate_sql"
    QUERY_DATA = "query_data"
    COMPLETE = "complete"

# Data Spec: Context for both workflows
class DataContext(BaseModel):
    user_input: str
    answer: Optional[str] = Field(default=None)
    sql_query: Optional[str] = Field(default=None)
    query_result: Optional[dict] = Field(default=None)
    execution_trace: List[str] = Field(default_factory=list)

# Contract: Base state interface
class BaseState(ABC):
    @abstractmethod
    def execute(self, context: DataContext) -> State:
        pass

# Namespaced Registry System
class StateRegistry:
    _registries = {}
    
    @classmethod
    def get_registry(cls, name: str):
        if name not in cls._registries:
            cls._registries[name] = {}
        return cls._registries[name]
    
    @classmethod
    def register(cls, registry_name: str, state_enum_value: State, state_class):
        registry = cls.get_registry(registry_name)
        registry[state_enum_value] = state_class
    
    @classmethod
    def get(cls, registry_name: str, state_enum_value: State):
        registry = cls.get_registry(registry_name)
        return registry[state_enum_value]()

def state(registry_name: str, state_enum_value: State):
    def decorator(cls):
        StateRegistry.register(registry_name, state_enum_value, cls)
        return cls
    return decorator

class StateMachine:
    def __init__(self, registry_name: str, start_state: State, complete_state: State, context_class):
        self.registry_name = registry_name
        self.start_state = start_state
        self.complete_state = complete_state
        self.context_class = context_class
    
    def run(self, **initial_data) -> DataContext:
        context = self.context_class(**initial_data)
        current_state = self.start_state
        
        while current_state != self.complete_state:
            context.execution_trace.append(current_state.value)
            state_instance = StateRegistry.get(self.registry_name, current_state)
            current_state = state_instance.execute(context)
        
        return context

# ===== CHAT WORKFLOW =====
@state("chat_workflow", State.ROUTER)
class ChatRouter(BaseState):
    def execute(self, context: DataContext) -> State:
        return State.CHAT

@state("chat_workflow", State.CHAT)
class ChatHandler(BaseState):
    def execute(self, context: DataContext) -> State:
        context.answer = f"Chat response: {context.user_input}"
        return State.COMPLETE

# ===== SQL WORKFLOW =====
@state("sql_workflow", State.ROUTER)
class SQLRouter(BaseState):
    def execute(self, context: DataContext) -> State:
        return State.GENERATE_SQL

@state("sql_workflow", State.GENERATE_SQL)
class SQLGenerator(BaseState):
    def execute(self, context: DataContext) -> State:
        context.sql_query = f"SELECT * FROM data WHERE query LIKE '%{context.user_input}%'"
        return State.QUERY_DATA

@state("sql_workflow", State.QUERY_DATA)
class DataQuerier(BaseState):
    def execute(self, context: DataContext) -> State:
        context.query_result = {"rows": 5, "data": "mock_data"}
        context.answer = f"SQL: {context.sql_query} | Result: {context.query_result}"
        return State.COMPLETE

# ===== USAGE =====
# Create separate machines
chat_machine = StateMachine("chat_workflow", State.ROUTER, State.COMPLETE, DataContext)
sql_machine = StateMachine("sql_workflow", State.ROUTER, State.COMPLETE, DataContext)

# Test Chat Workflow
print("=== CHAT WORKFLOW ===")
chat_result = chat_machine.run(user_input="hello world")
print(f"Answer: {chat_result.answer}")
print(f"Trace: {chat_result.execution_trace}")

# Test SQL Workflow  
print("\n=== SQL WORKFLOW ===")
sql_result = sql_machine.run(user_input="show sales data")
print(f"Answer: {sql_result.answer}")
print(f"SQL Query: {sql_result.sql_query}")
print(f"Trace: {sql_result.execution_trace}")

# Combined Usage
def combined_workflow(user_input: str):
    if "sql" in user_input.lower() or "data" in user_input.lower():
        return sql_machine.run(user_input=user_input)
    else:
        return chat_machine.run(user_input=user_input)

print("\n=== COMBINED WORKFLOW ===")
result1 = combined_workflow("show me data")
print(f"SQL Result: {result1.answer}")

result2 = combined_workflow("hello there")
print(f"Chat Result: {result2.answer}")


=== CHAT WORKFLOW ===
Answer: Chat response: hello world
Trace: ['router', 'chat']

=== SQL WORKFLOW ===
Answer: SQL: SELECT * FROM data WHERE query LIKE '%show sales data%' | Result: {'rows': 5, 'data': 'mock_data'}
SQL Query: SELECT * FROM data WHERE query LIKE '%show sales data%'
Trace: ['router', 'generate_sql', 'query_data']

=== COMBINED WORKFLOW ===
SQL Result: SQL: SELECT * FROM data WHERE query LIKE '%show me data%' | Result: {'rows': 5, 'data': 'mock_data'}
Chat Result: Chat response: hello there


In [18]:
from enum import Enum
from pydantic import BaseModel, Field
from typing import Optional, List
from abc import ABC, abstractmethod

# Blueprint: Single workflow with conditional paths
class State(Enum):
    ROUTER = "router"
    CHAT = "chat"
    SQL = "sql"
    COMPLETE = "complete"

# Data Spec: Context with routing info
class DataContext(BaseModel):
    user_input: str
    answer: Optional[str] = Field(default=None)
    sql_query: Optional[str] = Field(default=None)
    route_type: Optional[str] = Field(default=None)  # Track routing decision
    execution_trace: List[str] = Field(default_factory=list)

# Contract: Base state interface
class BaseState(ABC):
    @abstractmethod
    def execute(self, context: DataContext) -> State:
        pass

# Simple Registry (single workflow)
class StateRegistry:
    _states = {}
    
    @classmethod
    def register(cls, state_enum_value: State, state_class):
        cls._states[state_enum_value] = state_class
    
    @classmethod
    def get(cls, state_enum_value: State):
        return cls._states[state_enum_value]()

def state(state_enum_value: State):
    def decorator(cls):
        StateRegistry.register(state_enum_value, cls)
        return cls
    return decorator

class StateMachine:
    def __init__(self, start_state: State, complete_state: State, context_class):
        self.start_state = start_state
        self.complete_state = complete_state
        self.context_class = context_class
    
    def run(self, **initial_data) -> DataContext:
        context = self.context_class(**initial_data)
        current_state = self.start_state
        
        while current_state != self.complete_state:
            context.execution_trace.append(current_state.value)
            state_instance = StateRegistry.get(current_state)
            current_state = state_instance.execute(context)
        
        return context

# ===== WORKFLOW STATES =====
@state(State.ROUTER)
class Router(BaseState):
    def execute(self, context: DataContext) -> State:
        if "sql" in context.user_input.lower() or "data" in context.user_input.lower():
            context.route_type = "sql"
            return State.SQL
        else:
            context.route_type = "chat"
            return State.CHAT

@state(State.SQL)
class SQLProcessor(BaseState):
    def execute(self, context: DataContext) -> State:
        context.sql_query = f"SELECT * FROM data WHERE query LIKE '%{context.user_input}%'"
        # After SQL processing, go to chat for response formatting
        return State.CHAT

@state(State.CHAT)
class ChatHandler(BaseState):
    def execute(self, context: DataContext) -> State:
        if context.route_type == "sql":
            # Format SQL response
            context.answer = f"SQL Query: {context.sql_query}\nResult: Found 5 matching records"
        else:
            # Regular chat response
            context.answer = f"Chat response: {context.user_input}"
        
        return State.COMPLETE

# ===== USAGE =====
machine = StateMachine(State.ROUTER, State.COMPLETE, DataContext)

# Test direct chat: user_input -> router -> chat -> complete
print("=== DIRECT CHAT ===")
chat_result = machine.run(user_input="hello world")
print(f"Answer: {chat_result.answer}")
print(f"Route: {chat_result.route_type}")
print(f"Trace: {chat_result.execution_trace}")

# Test SQL flow: user_input -> router -> sql -> chat -> complete
print("\n=== SQL FLOW ===")
sql_result = machine.run(user_input="show me data")
print(f"Answer: {sql_result.answer}")
print(f"SQL Query: {sql_result.sql_query}")
print(f"Route: {sql_result.route_type}")
print(f"Trace: {sql_result.execution_trace}")


=== DIRECT CHAT ===
Answer: Chat response: hello world
Route: chat
Trace: ['router', 'chat']

=== SQL FLOW ===
Answer: SQL Query: SELECT * FROM data WHERE query LIKE '%show me data%'
Result: Found 5 matching records
SQL Query: SELECT * FROM data WHERE query LIKE '%show me data%'
Route: sql
Trace: ['router', 'sql', 'chat']


In [19]:
# Test individual states
def test_router():
    context = DataContext(user_input="sql query")
    router = Router()
    next_state = router.execute(context)
    assert next_state == State.SQL

# Test full workflow
def test_sql_flow():
    machine = StateMachine(State.ROUTER, State.COMPLETE, DataContext)
    result = machine.run(user_input="show data")
    assert "SQL Query:" in result.answer
    assert result.execution_trace == ['router', 'sql', 'chat']


In [20]:
test_router()

In [21]:
test_sql_flow()